In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from transformers import ModernBertConfig
from tqdm import tqdm






/home/hice1/kjacob7/.conda/envs/eml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_train = pd.read_csv("scratch/4641_data/train_dataset.csv")
print(df_train["text_length"].max())
df_train = df_train[["text", "sentiment"]]
df_train["sentiment"] = df_train["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})

df_val = pd.read_csv("scratch/4641_data/val_dataset.csv")
df_val = df_val[["text", "sentiment"]]
df_val["sentiment"] = df_val["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})

df_test = pd.read_csv("scratch/4641_data/test_dataset.csv")
df_test = df_test[["text", "sentiment", "bucket"]]
df_test["sentiment"] = df_test["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})


12930


In [3]:
num_classes = 3

In [4]:
class SentimentDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=3000):
        self.bodies = data["text"].values
        self.labels = data["sentiment"].values
        self.buckets = data["bucket"].values if "bucket" in data.columns else None
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.bodies)
    def __getitem__(self, idx):
        encoded = self.tokenizer(str(self.bodies[idx]), truncation=True, padding=False, max_length = self.max_length, return_tensors = "pt")
        return {"input_ids": encoded["input_ids"].squeeze(), "attention_mask": encoded["attention_mask"].squeeze(), "label": torch.tensor(self.labels[idx])}

In [5]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
config = ModernBertConfig.from_pretrained("answerdotai/ModernBERT-base", reference_compile=False)
bert_model = AutoModel.from_pretrained("answerdotai/ModernBERT-base", config=config)

In [6]:
train_data = SentimentDataset(df_train, tokenizer)
val_data = SentimentDataset(df_val, tokenizer)
test_data = SentimentDataset(df_test, tokenizer)

In [7]:

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_data, batch_size=16, collate_fn=collator)
test_loader = DataLoader(test_data, batch_size=16, collate_fn=collator)

In [8]:
df_train["token_count"] = df_train["text"].apply(lambda x: len(tokenizer.encode(x)))
print(df_train["token_count"].describe())


count    41999.000000
mean        97.454368
std        142.511286
min          3.000000
25%         20.000000
50%         42.000000
75%        109.000000
max       2907.000000
Name: token_count, dtype: float64


In [9]:
print(len(train_data))
print(len(val_data))
print(len(test_data))

41999
6000
12000


In [10]:
class bertSentimentAnalyzer(nn.Module):
    def __init__ (self, num_classes=num_classes, freeze_base=True):
        super().__init__()
        self.base = bert_model
        self.dropout = nn.Dropout(0.3)
        self.linear1 = nn.Linear(self.base.config.hidden_size, num_classes)
        if freeze_base:
            for param in self.base.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        out = out.last_hidden_state[: , 0, :]
        out = self.dropout(out)
        out = self.linear1(out)
        return out

In [11]:

model = bertSentimentAnalyzer(freeze_base=True)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
#optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 10
training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*training_steps),num_training_steps=training_steps)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(device)
model.to(device)
print(device)


cuda


In [ ]:

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels


best_val_loss = float("inf")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print("Saved best model!")


model.load_state_dict(torch.load("best_model.pt"))
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
print(f"\nTest Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

from sklearn.metrics import classification_report
print(classification_report(test_labels, test_preds, target_names=["negative", "neutral", "positive"]))

import pandas as pd
results_df = pd.DataFrame({
    "label": test_labels,
    "pred": test_preds,
    "bucket": df_test["bucket"].values
})

for bucket in ["super_short", "short", "medium", "long"]:
    subset = results_df[results_df["bucket"] == bucket]
    if len(subset) == 0:
        continue
    print(f"\n--- Bucket: {bucket} (n={len(subset)}) ---")
    print(classification_report(subset["label"], subset["pred"],
                                 target_names=["negative", "neutral", "positive"],
                                 zero_division=0))

from sklearn.metrics import classification_report
print(classification_report(test_labels, test_preds, target_names=["negative", "neutral", "positive"]))


Epoch 1/10


Evaluating: 100%|██████████| 375/375 [00:12<00:00, 30.59it/s]


Train Loss: 0.7284 | Train Acc: 0.6749
Val Loss:   0.4736 | Val Acc:   0.8115
Saved best model!

Epoch 2/10


Evaluating: 100%|██████████| 375/375 [00:12<00:00, 30.58it/s]


Train Loss: 0.4061 | Train Acc: 0.8452
Val Loss:   0.4274 | Val Acc:   0.8432
Saved best model!

Epoch 3/10


...

Train Loss: 0.0170 | Train Acc: 0.9929
Val Loss:   1.1967 | Val Acc:   0.8683

Epoch 10/10


Evaluating: 100%|██████████| 375/375 [00:12<00:00, 30.55it/s]


Train Loss: 0.0152 | Train Acc: 0.9932
Val Loss:   1.2138 | Val Acc:   0.8663


Evaluating: 100%|██████████| 750/750 [00:25<00:00, 29.90it/s]


Test Loss: 0.4261 | Test Acc: 0.8398
              precision    recall  f1-score   support

    negative       0.84      0.87      0.86      4050
     neutral       0.79      0.75      0.77      3451
    positive       0.87      0.88      0.87      4499

    accuracy                           0.84     12000
   macro avg       0.84      0.83      0.83     12000
weighted avg       0.84      0.84      0.84     12000


--- Bucket: super_short (n=4000) ---
              precision    recall  f1-score   support

    negative       0.83      0.80      0.81      1184
     neutral       0.72      0.76      0.74      1170
    positive       0.86      0.84      0.85      1646

    accuracy                           0.80      4000
   macro avg       0.80      0.80      0.80      4000
weighted avg       0.81      0.80      0.81      4000


--- Bucket: short (n=4001) ---
              precision    recall  f1-score   support

    negative       0.84      0.86      0.85      1361
     neutral       0.

In [16]:
%%bash
nvidia-smi

Sat Apr 11 20:13:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:19:00.0 Off |                    0 |
| N/A   59C    P0            138W /  700W |  116199MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----